# 🔬 Breast Cancer MLOps Demo — with MLflow Tracking
**Dr. Priya Lakshmi Narayanan** · [pathdata/LearningCurve](https://github.com/pathdata/LearningCurve/blob/master/ML/ch03_breast_cancer_v2.py)

This notebook extends `ch03_breast_cancer_v2.py` with **MLflow experiment tracking**:
- `mlflow.log_metric()` — records accuracy, F1, ROC-AUC, PR-AUC per model per run
- `mlflow.log_artifact()` — saves confusion matrix, ROC, PR curve PNGs to the run
- `mlflow.log_params()` — records hyperparameters (C, gamma, k, etc.)
- `mlflow.sklearn.log_model()` — saves the entire fitted imblearn pipeline

> **Google Colab note:** MLflow runs are stored locally inside the Colab session. We use `mlflow.set_tracking_uri("file:/content/drive/MyDrive/mlrunsmlruns")` and the MLflow UI is tunnelled via `ngrok` (optional cell at the end).


## ⚙️ Cell 1 — Install dependencies

In [1]:
# Install all required packages
!pip install scikit-learn imbalanced-learn mlflow joblib matplotlib numpy -q
print("✓ All packages installed")


✓ All packages installed


## 📦 Cell 2 — Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')           # non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve,
    make_scorer
)

# imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# MLflow
import mlflow
import mlflow.sklearn

print("✓ Imports complete")
print(f"  scikit-learn : {__import__('sklearn').__version__}")
print(f"  imbalanced-learn: {__import__('imblearn').__version__}")
print(f"  mlflow       : {mlflow.__version__}")


✓ Imports complete
  scikit-learn : 1.6.1
  imbalanced-learn: 0.14.2
  mlflow       : 3.13.0


## 🗂️ Cell 3 — MLflow Experiment Setup

MLflow tracks every training run inside an **experiment**. Each run stores:
- **Parameters** — hyperparameters (C, gamma, k_neighbors, …)
- **Metrics** — accuracy, F1, ROC-AUC, PR-AUC, recall
- **Artifacts** — PNG plots, saved model pipeline, metadata text


In [5]:
#
import os

DB_PATH         = "/content/drive/MyDrive/mlruns/mlflow.db"
TRACKING_URI    = f"sqlite:///{DB_PATH}"
EXPERIMENT_NAME = "breast_cancer_smote_classifiers"

# Ensure the folder exists before SQLite creates the file
os.makedirs("/content/drive/MyDrive/mlruns", exist_ok=True)

mlflow.set_tracking_uri(TRACKING_URI)

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"✓ MLflow experiment   : {EXPERIMENT_NAME}")
print(f"  Tracking URI        : {TRACKING_URI}")
print(f"  Experiment ID       : {exp.experiment_id}")
print(f"  DB file             : {DB_PATH}")

2026/06/16 09:23:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/16 09:23:58 INFO mlflow.store.db.utils: Updating database tables


✓ MLflow experiment   : breast_cancer_smote_classifiers
  Tracking URI        : sqlite:////content/drive/MyDrive/mlruns/mlflow.db
  Experiment ID       : 1
  DB file             : /content/drive/MyDrive/mlruns/mlflow.db


## 📊 Cell 4 — Data Loading & Class Imbalance

**Dataset:** Breast Cancer Wisconsin (569 samples, 30 features)
- Label 0 = malignant (~37%) ← minority class
- Label 1 = benign   (~63%)

`stratify=y` ensures both splits have the same malignant/benign ratio.


In [6]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
feature_names = bc.feature_names
target_names  = bc.target_names   # ['malignant', 'benign']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

counts_train = np.bincount(y_train)
counts_test  = np.bincount(y_test)

print(f"Dataset : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes : {list(target_names)} (0=malignant, 1=benign)\n")
print(f"{'Split':<8} {'Total':>6} {'Malignant':>10} {'Benign':>8} {'Ratio':>10}")
print("-" * 46)
print(f"{'Train':<8} {X_train.shape[0]:>6} {counts_train[0]:>10} {counts_train[1]:>8}  1:{counts_train[1]/counts_train[0]:.2f}")
print(f"{'Test':<8} {X_test.shape[0]:>6}  {counts_test[0]:>10} {counts_test[1]:>8}  1:{counts_test[1]/counts_test[0]:.2f}")


Dataset : 569 samples, 30 features
Classes : [np.str_('malignant'), np.str_('benign')] (0=malignant, 1=benign)

Split     Total  Malignant   Benign      Ratio
----------------------------------------------
Train       398        148      250  1:1.69
Test        171          64      107  1:1.67


## 🔧 Cell 5 — Build imblearn Pipelines

`imblearn.pipeline.Pipeline` applies SMOTE **only during `fit()`**, never during `predict()`.
This means the test set is never resampled — no data leakage.


In [7]:
smote = SMOTE(random_state=42, k_neighbors=5)

pipe_lr = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote",  smote),
    ("clf",    LogisticRegression(C=1.0, max_iter=500, random_state=42))
])

pipe_knn = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote",  smote),
    ("clf",    KNeighborsClassifier(n_neighbors=7, metric="euclidean", weights="distance"))
])

pipe_rf = ImbPipeline([
    ("smote", smote),
    ("clf",   RandomForestClassifier(n_estimators=500, max_features="sqrt",
                                      criterion="gini", random_state=42, n_jobs=-1))
])

pipe_svm_default = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote",  smote),
    ("clf",    SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42, probability=True))
])

print("Pipelines built:")
for name in ["Logistic Regression", "k-NN (k=7)", "Random Forest", "SVM (default)"]:
    print(f"  StandardScaler → SMOTE → {name}")

print("\nFitting pipelines …")
pipe_lr.fit(X_train, y_train)
pipe_knn.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
pipe_svm_default.fit(X_train, y_train)
print("✓ Done")


Pipelines built:
  StandardScaler → SMOTE → Logistic Regression
  StandardScaler → SMOTE → k-NN (k=7)
  StandardScaler → SMOTE → Random Forest
  StandardScaler → SMOTE → SVM (default)

Fitting pipelines …
✓ Done


## 🔍 Cell 6 — SVM GridSearchCV

Exhaustive search over C × gamma. Scoring = F1 for the malignant class (pos_label=0).
SMOTE runs independently inside each of the 5 CV folds — no leakage.


In [8]:
cv           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_malignant = make_scorer(f1_score, pos_label=0)

C_grid     = [0.01, 0.1, 1, 10, 100]
gamma_grid = [0.001, 0.01, 0.1, 1, "scale"]

svm_gs_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote",  SMOTE(random_state=42, k_neighbors=5)),
    ("clf",    SVC(kernel="rbf", random_state=42, probability=True))
])

gs_svm = GridSearchCV(
    svm_gs_pipe,
    {"clf__C": C_grid, "clf__gamma": gamma_grid},
    scoring=f1_malignant, cv=cv, n_jobs=-1, verbose=0,
    return_train_score=True
)
gs_svm.fit(X_train, y_train)
pipe_svm = gs_svm.best_estimator_

print(f"Best params  : {gs_svm.best_params_}")
print(f"Best CV F1   : {gs_svm.best_score_:.4f}")


Best params  : {'clf__C': 100, 'clf__gamma': 0.001}
Best CV F1   : 0.9754


## 🎨 Cell 7 — Plot Helpers (saved as PNG for MLflow artifacts)

In [10]:
# os.makedirs("mlflow_artifacts", exist_ok=True)
os.makedirs("/content/drive/MyDrive/mlflow_artifacts", exist_ok=True)

# ── GridSearch heatmap ────────────────────────────────────────────────────────
results      = gs_svm.cv_results_
mean_test    = results["mean_test_score"].reshape(len(C_grid), len(gamma_grid))
std_test     = results["std_test_score"].reshape(len(C_grid), len(gamma_grid))
mean_train   = results["mean_train_score"].reshape(len(C_grid), len(gamma_grid))
gamma_labels = [str(g) for g in gamma_grid]
C_labels     = [str(c) for c in C_grid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SVM GridSearchCV — F1 (malignant class), 5-fold CV", fontsize=12, fontweight="bold")
for ax, matrix, title, cmap in zip(
        axes, [mean_test, mean_train],
        ["Mean CV Test F1", "Mean CV Train F1"], ["YlOrRd", "YlGnBu"]):
    im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0.7, vmax=1.0)
    ax.set_xticks(range(len(gamma_labels))); ax.set_xticklabels(gamma_labels, fontsize=9)
    ax.set_yticks(range(len(C_labels)));     ax.set_yticklabels(C_labels, fontsize=9)
    ax.set_xlabel("gamma"); ax.set_ylabel("C"); ax.set_title(title, fontsize=11)
    for i in range(len(C_grid)):
        for j in range(len(gamma_grid)):
            val = matrix[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8,
                    color="white" if val > 0.90 else "black", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
best_i = C_grid.index(gs_svm.best_params_["clf__C"])
best_j = gamma_labels.index(str(gs_svm.best_params_["clf__gamma"]))
axes[0].add_patch(plt.Rectangle((best_j-0.5, best_i-0.5), 1, 1,
    fill=False, edgecolor="lime", linewidth=3, label="Best params"))
axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
# HEATMAP_PATH = "mlflow_artifacts/svm_gridsearch_heatmap.png"
HEATMAP_PATH = "/content/drive/MyDrive/mlflow_artifacts/svm_gridsearch_heatmap.png"
plt.savefig(HEATMAP_PATH, dpi=120, bbox_inches="tight")
plt.show(); plt.close()
print(f"Saved: {HEATMAP_PATH}")


Saved: /content/drive/MyDrive/mlflow_artifacts/svm_gridsearch_heatmap.png


## 📈 Cell 8 — Evaluate All Classifiers & Generate Plot Artifacts

Computes all metrics and saves PNG plots that will be uploaded as MLflow artifacts.


In [11]:
classifiers = {
    "Logistic Regression": pipe_lr,
    "k-NN (k=7)":          pipe_knn,
    "Random Forest":       pipe_rf,
    "SVM (tuned)":         pipe_svm,
}

results_table   = {}
baseline_prec   = np.mean(y_test == 0)
colors          = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]

print(f"\n{'Classifier':<22} {'Acc':>6} {'F1-M':>6} {'ROC-AUC':>8} {'PR-AUC':>7} {'Recall-0':>9}")
print("  " + "─" * 62)

for name, clf in classifiers.items():
    y_pred  = clf.predict(X_test)
    y_prob  = clf.predict_proba(X_test)[:, 0]   # P(malignant)
    acc     = accuracy_score(y_test, y_pred)
    f1_m    = f1_score(y_test, y_pred, pos_label=0)
    rec_m   = recall_score(y_test, y_pred, pos_label=0)
    roc_auc = roc_auc_score(1 - y_test, y_prob)
    pr_auc  = average_precision_score(1 - y_test, y_prob)
    results_table[name] = dict(acc=acc, f1_m=f1_m, roc_auc=roc_auc,
                                pr_auc=pr_auc, recall_m=rec_m,
                                clf=clf, y_prob=y_prob)
    print(f"  {name:<22} {acc:>6.3f} {f1_m:>6.3f} {roc_auc:>8.3f} {pr_auc:>7.3f} {rec_m:>9.3f}")

best_name = max(results_table, key=lambda k: results_table[k]["f1_m"])
print(f"\n✓ Best model by F1-malignant: {best_name} (F1={results_table[best_name]['f1_m']:.3f})")

# ── Confusion matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Confusion Matrices — Breast Cancer Test Set (0=malignant, 1=benign)",
             fontsize=11, fontweight="bold")
for ax, (name, clf) in zip(axes, classifiers.items()):
    ConfusionMatrixDisplay.from_estimator(
        clf, X_test, y_test, display_labels=target_names,
        colorbar=False, ax=ax, cmap="Blues")
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.add_patch(plt.Rectangle((0.5, -0.5), 1, 1, fill=False,
                                edgecolor="red", linewidth=2))
axes[-1].add_patch(plt.Rectangle((0.5, -0.5), 1, 1, fill=False,
                                  edgecolor="red", linewidth=2, label="FN (missed malignant)"))
axes[-1].legend(loc="upper right", fontsize=7)
plt.tight_layout()
CM_PATH = "/content/drive/MyDrive/mlflow_artifacts/confusion_matrices.png"
plt.savefig(CM_PATH, dpi=120, bbox_inches="tight")
plt.show(); plt.close()

# ── ROC curves ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for (name, res), color in zip(results_table.items(), colors):
    fpr, tpr, _ = roc_curve(1 - y_test, res["y_prob"])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1], "k--", linewidth=1, label="Chance (AUC=0.50)")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR (Sensitivity)")
ax.set_title("ROC Curves — Breast Cancer (positive = malignant)", fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
ROC_PATH = "/content/drive/MyDrive/mlflow_artifacts/roc_curves.png"
plt.savefig(ROC_PATH, dpi=120, bbox_inches="tight")
plt.show(); plt.close()

# ── PR curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for (name, res), color in zip(results_table.items(), colors):
    prec, rec, _ = precision_recall_curve(1 - y_test, res["y_prob"])
    ax.plot(rec, prec, color=color, linewidth=2, label=f"{name} (AP={res['pr_auc']:.3f})")
ax.axhline(baseline_prec, linestyle="--", color="grey", linewidth=1.2,
           label=f"No-skill baseline ({baseline_prec:.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves (positive = malignant)", fontweight="bold")
ax.legend(loc="upper right", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
PR_PATH = "/content/drive/MyDrive/mlflow_artifacts/pr_curves.png"
plt.savefig(PR_PATH, dpi=120, bbox_inches="tight")
plt.show(); plt.close()

print(f"\n✓ Plots saved: {CM_PATH}, {ROC_PATH}, {PR_PATH}")



Classifier                Acc   F1-M  ROC-AUC  PR-AUC  Recall-0
  ──────────────────────────────────────────────────────────────
  Logistic Regression     0.965  0.955    0.998   0.997     0.984
  k-NN (k=7)              0.953  0.938    0.993   0.989     0.938
  Random Forest           0.953  0.938    0.994   0.991     0.953
  SVM (tuned)             0.982  0.976    0.995   0.993     0.969

✓ Best model by F1-malignant: SVM (tuned) (F1=0.976)

✓ Plots saved: /content/drive/MyDrive/mlflow_artifacts/confusion_matrices.png, /content/drive/MyDrive/mlflow_artifacts/roc_curves.png, /content/drive/MyDrive/mlflow_artifacts/pr_curves.png


## 🧪 Cell 9 — MLflow: Log Metrics, Params & Artifacts

This is the core MLOps cell. Each classifier gets its **own MLflow run** so results are independently queryable.

### What gets logged per run:
| Call | What it stores |
|------|---------------|
| `mlflow.log_param()` | Model name, hyperparameters, SMOTE k |
| `mlflow.log_metric()` | accuracy, F1-malignant, ROC-AUC, PR-AUC, recall |
| `mlflow.log_artifact()` | PNG plots (confusion matrix, ROC, PR) |
| `mlflow.sklearn.log_model()` | Full fitted imblearn Pipeline |
| `mlflow.set_tag()` | Best-model flag, SMOTE strategy |


In [12]:
# ── Helper: extract hyperparams from an imblearn pipeline ────────────────────
def extract_params(name, clf_pipeline):
    """Pull hyperparameters out of the pipeline's classifier step."""
    p = {}
    try:
        estimator = clf_pipeline.named_steps["clf"]
        if hasattr(estimator, "C"):          p["C"]           = estimator.C
        if hasattr(estimator, "gamma"):      p["gamma"]       = str(estimator.gamma)
        if hasattr(estimator, "kernel"):     p["kernel"]      = estimator.kernel
        if hasattr(estimator, "n_neighbors"):p["n_neighbors"] = estimator.n_neighbors
        if hasattr(estimator, "n_estimators"):p["n_estimators"]= estimator.n_estimators
        if hasattr(estimator, "max_features"):p["max_features"]= str(estimator.max_features)
        if hasattr(estimator, "max_iter"):   p["max_iter"]    = estimator.max_iter
    except Exception:
        pass
    # SMOTE params
    try:
        smote_step = clf_pipeline.named_steps["smote"]
        p["smote_k_neighbors"] = smote_step.k_neighbors
        p["smote_random_state"] = smote_step.random_state
    except Exception:
        pass
    return p


# ── Log every classifier as a separate MLflow run ────────────────────────────
run_ids = {}

for name, res in results_table.items():
    clf     = res["clf"]
    metrics = {
        "accuracy":     res["acc"],
        "f1_malignant": res["f1_m"],
        "roc_auc":      res["roc_auc"],
        "pr_auc":       res["pr_auc"],
        "recall_malignant": res["recall_m"],
    }
    params = {
        "model_name":      name,
        "test_size":       0.30,
        "random_state":    42,
        "smote_strategy":  "minority_oversampling",
        **extract_params(name, clf),
    }

    with mlflow.start_run(run_name=name) as run:
        run_ids[name] = run.info.run_id

        # ── 1. Log hyperparameters ─────────────────────────────────────
        mlflow.log_params(params)

        # ── 2. Log evaluation metrics ──────────────────────────────────
        mlflow.log_metrics(metrics)

        # ── 3. Log PNG plot artifacts ──────────────────────────────────
        mlflow.log_artifact(CM_PATH,      artifact_path="plots")
        mlflow.log_artifact(ROC_PATH,     artifact_path="plots")
        mlflow.log_artifact(PR_PATH,      artifact_path="plots")
        mlflow.log_artifact(HEATMAP_PATH, artifact_path="plots")

        # ── 4. Log the fitted pipeline (scaler + SMOTE + classifier) ───
        mlflow.sklearn.log_model(
            clf,
            artifact_path="pipeline",
            registered_model_name=f"bc_{name.lower().replace(' ','_').replace('(','').replace(')','').replace('=','')}",
        )

        # ── 5. Tags ────────────────────────────────────────────────────
        mlflow.set_tag("is_best_model",   str(name == best_name))
        mlflow.set_tag("smote",           "imblearn.pipeline — no leakage")
        mlflow.set_tag("positive_class",  "malignant (label=0)")

        print(f"  ✓ {name:<22}  run_id={run.info.run_id[:8]}…  "
              f"F1={res['f1_m']:.3f}  AUC={res['roc_auc']:.3f}")

print(f"\n✓ All runs logged to experiment '{EXPERIMENT_NAME}'")


2026/06/16 09:38:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:38:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'bc_logistic_regression'.
Created version '1' of model 'bc_logistic_regression'.
2026/06/16 09:39:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✓ Logistic Regression     run_id=a9417af8…  F1=0.955  AUC=0.998


2026/06/16 09:39:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'bc_k-nn_k7'.
Created version '1' of model 'bc_k-nn_k7'.
2026/06/16 09:39:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✓ k-NN (k=7)              run_id=e624b250…  F1=0.938  AUC=0.993


2026/06/16 09:39:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'bc_random_forest'.
Created version '1' of model 'bc_random_forest'.
2026/06/16 09:39:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✓ Random Forest           run_id=dc4d5a81…  F1=0.938  AUC=0.994


2026/06/16 09:39:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ SVM (tuned)             run_id=8ed274c4…  F1=0.976  AUC=0.995

✓ All runs logged to experiment 'breast_cancer_smote_classifiers'


Successfully registered model 'bc_svm_tuned'.
Created version '1' of model 'bc_svm_tuned'.


## 💾 Cell 10 — Save Best Model & Log as MLflow Artifact

The best-performing pipeline is saved locally **and** registered in MLflow's model registry.


In [13]:
os.makedirs("/content/drive/MyDrive/mlflow_artifacts/saved_models", exist_ok=True)

best_clf     = results_table[best_name]["clf"]
best_metrics = results_table[best_name]

JOBLIB_PATH  = "/content/drive/MyDrive/mlflow_artifacts/saved_models/best_model.joblib"
PKL_PATH     = "/content/drive/MyDrive/mlflow_artifacts/saved_models/best_model.pkl"
META_PATH    = "/content/drive/MyDrive/mlflow_artifacts/saved_models/model_metadata.txt"

# ── Local saves ───────────────────────────────────────────────────────────────
joblib.dump(best_clf, JOBLIB_PATH)
with open(PKL_PATH, "wb") as f:
    pickle.dump(best_clf, f)

metadata = f"""Best Model — Breast Cancer Classifier
======================================
Model          : {best_name}
MLflow run ID  : {run_ids[best_name]}
Experiment     : {EXPERIMENT_NAME}

Test-set metrics (n={X_test.shape[0]}):
  Accuracy        : {best_metrics['acc']:.4f}
  F1 (malignant)  : {best_metrics['f1_m']:.4f}
  ROC-AUC         : {best_metrics['roc_auc']:.4f}
  PR-AUC          : {best_metrics['pr_auc']:.4f}
  Recall-0        : {best_metrics['recall_m']:.4f}

Reload (joblib):
  import joblib
  model = joblib.load('saved_models/best_model.joblib')
  y_pred = model.predict(X_new)

Reload (MLflow):
  import mlflow
  model = mlflow.sklearn.load_model('runs:/{run_ids[best_name]}/pipeline')
  y_pred = model.predict(X_new)
"""
with open(META_PATH, "w") as f:
    f.write(metadata)

# ── Log saved files as MLflow artifacts on the best model's run ──────────────
with mlflow.start_run(run_id=run_ids[best_name]):
    mlflow.log_artifact(JOBLIB_PATH, artifact_path="saved_model")
    mlflow.log_artifact(PKL_PATH,    artifact_path="saved_model")
    mlflow.log_artifact(META_PATH,   artifact_path="saved_model")
    mlflow.set_tag("is_persisted", "True")

# ── Verify reload ─────────────────────────────────────────────────────────────
reloaded = joblib.load(JOBLIB_PATH)
assert np.array_equal(reloaded.predict(X_test), best_clf.predict(X_test)), "Reload mismatch!"
print(metadata)
print(f"✓ joblib reload verified — predictions identical")


Best Model — Breast Cancer Classifier
Model          : SVM (tuned)
MLflow run ID  : 8ed274c4c3734cdb90dc7e5a7d9561b5
Experiment     : breast_cancer_smote_classifiers

Test-set metrics (n=171):
  Accuracy        : 0.9825
  F1 (malignant)  : 0.9764
  ROC-AUC         : 0.9947
  PR-AUC          : 0.9927
  Recall-0        : 0.9688

Reload (joblib):
  import joblib
  model = joblib.load('saved_models/best_model.joblib')
  y_pred = model.predict(X_new)

Reload (MLflow):
  import mlflow
  model = mlflow.sklearn.load_model('runs:/8ed274c4c3734cdb90dc7e5a7d9561b5/pipeline')
  y_pred = model.predict(X_new)

✓ joblib reload verified — predictions identical


## 🔎 Cell 11 — Query MLflow Run Results Programmatically

Use the MLflow Python client to pull all run results into a DataFrame — useful for comparison and auditing.


In [14]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=TRACKING_URI)
exp    = client.get_experiment_by_name(EXPERIMENT_NAME)
runs   = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.f1_malignant DESC"]
)

print(f"{'Run name':<25} {'F1-M':>6} {'Acc':>6} {'ROC-AUC':>8} {'PR-AUC':>7}  {'Run ID'}")
print("─" * 80)
for r in runs:
    m = r.data.metrics
    print(f"  {r.data.tags.get('mlflow.runName',''):<23} "
          f"{m.get('f1_malignant',0):>6.3f} "
          f"{m.get('accuracy',0):>6.3f} "
          f"{m.get('roc_auc',0):>8.3f} "
          f"{m.get('pr_auc',0):>7.3f}  "
          f"{r.info.run_id[:12]}…")

print(f"\n✓ Best run: '{runs[0].data.tags.get('mlflow.runName')}' "
      f"(F1={runs[0].data.metrics.get('f1_malignant',0):.3f})")


Run name                    F1-M    Acc  ROC-AUC  PR-AUC  Run ID
────────────────────────────────────────────────────────────────────────────────
  SVM (tuned)              0.976  0.982    0.995   0.993  8ed274c4c373…
  Logistic Regression      0.955  0.965    0.998   0.997  a9417af8b846…
  Random Forest            0.938  0.953    0.994   0.991  dc4d5a816825…
  k-NN (k=7)               0.938  0.953    0.993   0.989  e624b2507f4a…

✓ Best run: 'SVM (tuned)' (F1=0.976)


## ♻️ Cell 12 — Reload Model Directly from MLflow

In production you would load the model from the MLflow registry — not from a local `.joblib` file.


In [15]:
best_run_id = runs[0].info.run_id

# Load the pipeline directly from the MLflow artifact store
model_uri   = f"runs:/{best_run_id}/pipeline"
loaded_clf  = mlflow.sklearn.load_model(model_uri)

y_pred_reloaded = loaded_clf.predict(X_test)
assert np.array_equal(y_pred_reloaded, best_clf.predict(X_test)), "MLflow reload mismatch!"

print(f"✓ Model reloaded from MLflow run: {best_run_id[:12]}…")
print(f"  URI          : {model_uri}")
print(f"  Predictions match local model: True")
print(f"  F1-malignant : {f1_score(y_test, y_pred_reloaded, pos_label=0):.4f}")
print(f"  ROC-AUC      : {roc_auc_score(1-y_test, loaded_clf.predict_proba(X_test)[:,0]):.4f}")


✓ Model reloaded from MLflow run: 8ed274c4c373…
  URI          : runs:/8ed274c4c3734cdb90dc7e5a7d9561b5/pipeline
  Predictions match local model: True
  F1-malignant : 0.9764
  ROC-AUC      : 0.9947


## 🌐 Cell 13 — (Optional) MLflow UI via ngrok

Launch the MLflow UI and tunnel it to the public internet using `pyngrok`.
This lets you browse runs, compare metrics, and inspect artifacts visually.

> **Skip this cell** if you don't need the UI — all results are available programmatically above.


In [ ]:
# Uncomment to install and launch the MLflow UI
# !pip install pyngrok -q

# from pyngrok import ngrok
# import subprocess, time

# # Kill any existing MLflow server
# !pkill -f "mlflow ui" 2>/dev/null || True
# time.sleep(1)

# # Start MLflow UI on port 5000
# proc = subprocess.Popen(
#     ["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000",
#      "--backend-store-uri", TRACKING_URI],
#     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
# )
# time.sleep(3)

# # Open public tunnel
# public_url = ngrok.connect(5000)
# print(f"✓ MLflow UI: {public_url}")
# print("  Browse runs → Artifacts → Plots / Saved models")

print("ℹ️  Uncomment the code above to launch the MLflow UI with a public ngrok URL.")
print("   All run data is already queryable via MlflowClient (Cell 11).")


## ✅ Cell 14 — Summary

### What this notebook demonstrates

| MLOps Principle | Implementation |
|----------------|---------------|
| **No data leakage** | SMOTE inside `ImbPipeline` — applied per CV fold, never on test set |
| **Reproducibility** | All `random_state=42`, pinned package versions |
| **Experiment tracking** | `mlflow.log_metric()` — 5 metrics per model per run |
| **Artifact management** | `mlflow.log_artifact()` — PNGs, `.joblib`, metadata per run |
| **Model registry** | `mlflow.sklearn.log_model()` — full pipeline registered |
| **Programmatic comparison** | `MlflowClient.search_runs()` — ranked leaderboard |
| **Reliable reload** | `mlflow.sklearn.load_model(runs:/{run_id}/pipeline)` |

### Output files
```
mlflow_artifacts/
  svm_gridsearch_heatmap.png
  confusion_matrices.png
  roc_curves.png
  pr_curves.png
saved_models/
  best_model.joblib
  best_model.pkl
  model_metadata.txt
mlruns/                         ← MLflow tracking store (all runs, metrics, artifacts)
```
